# 🛒 Ecommerce Analytics & Machine Learning (Google BigQuery & Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Geovana28/gcp-ecommerce-lakehouse/blob/main/notebooks/01_ecommerce_analytics_colab.ipynb)

Este notebook integra-se diretamente ao **Data Lakehouse no Google Cloud BigQuery** para realizar análises exploratórias avançadas, estatística inferencial de logística e modelagem preditiva de atraso na entrega de pedidos.

### 🎯 Objetivos de Análise:
1. **Autenticação e Conexão**: Conectar de forma nativa ao BigQuery pelo Colab.
2. **Logística & SLA**: Avaliar a distribuição empírica do tempo de entrega e desvio do prazo estimado.
3. **Modelagem Preditiva**: Treinar modelo de classificação para estimar risco de atraso (`is_delivery_delayed`).

## 1. Configuração e Autenticação no Google Cloud

In [ ]:
# No Google Colab, autentique sua conta Google associada ao GCP
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Autenticação realizada com sucesso no Google Cloud!")
except ImportError:
    print("ℹ️ Executando em ambiente local. Certifique-se de configurar GOOGLE_APPLICATION_CREDENTIALS.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

# Configuração do seu Project ID no Google Cloud
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "seu-projeto-gcp-id")
client = bigquery.Client(project=PROJECT_ID)

sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (10, 5)
print(f"BigQuery Client configurado para o projeto: {PROJECT_ID}")

## 2. Consulta à Camada Gold (Data Mart de Pedidos)

In [ ]:
# Query analítica extraindo os dados da camada Gold
query = f"""
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    delivery_time_days,
    delay_vs_estimate_days,
    is_delivery_delayed
FROM `{PROJECT_ID}.ecommerce_gold.fct_orders`
WHERE order_status = 'delivered'
  AND delivery_time_days IS NOT NULL
LIMIT 50000;
"""

# Execução com retorno direto para DataFrame Pandas
# Em caso de teste sem BigQuery conectado, carregamos uma amostra sintética
try:
    df_orders = client.query(query).to_dataframe()
    print(f"✅ Carregados {len(df_orders)} pedidos diretamente do BigQuery!")
except Exception as e:
    print(f"⚠️ Conexão BigQuery: {e}\nGerando amostra sintética para demonstração.")
    np.random.seed(42)
    n_samples = 1000
    delivery_time = np.random.gamma(shape=3, scale=4, size=n_samples)
    estimated_time = delivery_time + np.random.normal(loc=2, scale=3, size=n_samples)
    delays = np.maximum(0, delivery_time - estimated_time)
    df_orders = pd.DataFrame({
        'order_id': [f'ORD_{i:05d}' for i in range(n_samples)],
        'delivery_time_days': np.round(delivery_time, 1),
        'delay_vs_estimate_days': np.round(delays, 1),
        'is_delivery_delayed': delays > 0
    })

## 3. Análise Estatística de SLA & Entregas

In [ ]:
# Métricas Descritivas e Intervalo Interquartil (IQR)
total = len(df_orders)
atrasados = df_orders['is_delivery_delayed'].sum()
taxa_atraso = (atrasados / total) * 100
mediana_entrega = df_orders['delivery_time_days'].median()
p95_entrega = np.percentile(df_orders['delivery_time_days'], 95)

print(f"--- Sumário Operacional ---")
print(f"Total de pedidos analisados: {total:,}")
print(f"Taxa de atraso em relação ao prazo estimado: {taxa_atraso:.2f}%")
print(f"Mediana do tempo de entrega: {mediana_entrega:.1f} dias")
print(f"Percentil 95 (P95) de entrega: {p95_entrega:.1f} dias")

In [ ]:
# Distribuição do Tempo de Entrega (Histograma + KDE)
fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(df_orders['delivery_time_days'], kde=True, bins=35, color='#0284c7', ax=ax)
ax.axvline(mediana_entrega, color='#ef4444', linestyle='--', label=f'Mediana ({mediana_entrega:.1f} dias)')
ax.axvline(p95_entrega, color='#f59e0b', linestyle=':', label=f'P95 ({p95_entrega:.1f} dias)')
ax.set_title("Distribuição do Tempo Real de Entrega aos Clientes", fontsize=14, fontweight='bold')
ax.set_xlabel("Dias de Entrega")
ax.set_ylabel("Frequência de Pedidos")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Modelagem Preditiva de Atrasos (Scikit-Learn)

Treinamento de um classificador para calcular o risco de quebra de SLA logístico.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Engenharia simples de features para demonstração
X = df_orders[['delivery_time_days']]
y = df_orders['is_delivery_delayed'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
roc_score = roc_auc_score(y_test, y_pred_proba)

print(f"✅ Modelo treinado com sucesso!")
print(f"ROC-AUC Score: {roc_score:.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, (y_pred_proba > 0.5).astype(int), zero_division=0))